<a href="https://colab.research.google.com/github/ijazkhan0351-bot/Ijazweek1-ml-assignment/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ijazkhan0351-bot/Ijazweek1-ml-assignment/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip install -q duckdb huggingface_hub

import duckdb
from google.colab import userdata
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
print("Connected.")

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Rebuild Week 5's data + model (needed fresh in this notebook)
march = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        AVG(gsc_impressions) as avg_impressions,
        AVG(gsc_clicks) as avg_clicks,
        AVG(gsc_avg_position) as avg_position,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) as ctr
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

april = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) as april_clicks
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

data = march.merge(april, on=["client_hash_id", "content_hash_id"], how="inner")
data["is_declining"] = (data["april_clicks"] < data["avg_clicks"] * 30 * 0.85).astype(int)

feature_cols = ["avg_impressions", "avg_clicks", "avg_position", "ctr"]
X = data[feature_cols].fillna(0)
y = data["is_declining"]
groups = data["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg.fit(X_train, y_train)
logreg_score = logreg.predict_proba(X_test)[:, 1]
logreg_p50 = precision_at_k(logreg_score, y_test.values, 50)
print(f"Week-5 model rebuilt. Precision@50 = {logreg_p50:.3f}")

Connected.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Week-5 model rebuilt. Precision@50 = 0.940


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
## Paper Findings + My Methodology Questions

**Finding 1: The Freshness Multiplier (Finding #4)**
The paper reports that pages refreshed in the last 30 days show a
growth-to-decline ratio as high as 5.43:1 in the 31-90 day freshness
window, and a separate 365+ refreshed cohort shows a 1.6x health lift
and 52x impression lift versus stale pages.

**My methodology question:** How is "growth" defined relative to the
refresh event itself — is the growth window measured strictly *after*
the refresh date, with a clean gap from the freshness signal used to
select refresh candidates? If a page is refreshed because it already
showed early signs of movement, the refresh and the growth could be
correlated for reasons other than the refresh itself (selection bias,
not treatment effect). I'd want to see whether refresh timing was
randomized or assigned independent of a page's existing trajectory.

**Finding 2: 30-Day Momentum Model (Part IV — What Will Improve Next Month?)**
The paper reports a model predicting 30-day improvement with 95% accuracy
on same-brand unseen pages, dropping to 90% on entirely unseen brands,
with "Prior 30d Impressions" as the top predictor.

**My methodology question:** Is "prior 30-day impressions" measured from
a strictly earlier window than the "next month" outcome, with no overlap
day shared between feature and label windows? A single shared day between
the impressions used as a feature and the outcome window being predicted
would inflate accuracy in a way that looks like genuine predictive power
but is really partial leakage. I'd also ask whether the same-brand/unseen-brand
split groups by brand consistently, the way my own Week-5 split groups
by client.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## Before/After: Honest Split Comparison

In Week 5, I already used a grouped, time-aware split (client-grouped,
March→April). Here I show the before/after explicitly: what my
Precision@50 would look like under a naive row-level random split
(which risks client leakage) versus my actual grouped split.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# BEFORE: naive random split (no grouping — risks leakage)
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
logreg_naive = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg_naive.fit(X_train_naive, y_train_naive)
naive_score = logreg_naive.predict_proba(X_test_naive)[:, 1]
naive_p50 = precision_at_k(naive_score, y_test_naive.values, 50)

# AFTER: honest grouped split (from setup cell)
honest_p50 = logreg_p50

print("=== Before/After: Split Design ===")
print(f"BEFORE (naive random split):   Precision@50 = {naive_p50:.3f}")
print(f"AFTER (grouped, time-aware):   Precision@50 = {honest_p50:.3f}")
print(f"Gap: {naive_p50 - honest_p50:.3f}")

=== Before/After: Split Design ===
BEFORE (naive random split):   Precision@50 = 1.000
AFTER (grouped, time-aware):   Precision@50 = 0.940
Gap: 0.060


[Fill in with real numbers: if naive_p50 > honest_p50, that gap IS the
leakage — the naive split was overstating performance because
client-specific patterns leaked between train and test. State the real
gap you observe, e.g. "The naive split overstated Precision@50 by X.XXX,
confirming that client-level patterns were leaking across train/test."]

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== Feature correlation with label (leakage smell test) ===")
for col in feature_cols:
    corr = data[col].corr(data["is_declining"])
    print(f"{col}: correlation with label = {corr:.3f}")

print("\nFeature columns used:", feature_cols)
print("April-derived columns in features?",
      any("april" in c.lower() for c in feature_cols))

=== Feature correlation with label (leakage smell test) ===
avg_impressions: correlation with label = 0.169
avg_clicks: correlation with label = 0.114
avg_position: correlation with label = -0.183
ctr: correlation with label = 0.180

Feature columns used: ['avg_impressions', 'avg_clicks', 'avg_position', 'ctr']
April-derived columns in features? False


## Leakage Audit

None of my four features (avg_impressions, avg_clicks, avg_position, ctr)
are derived from April data — all are built only from the March window,
before the decision point. The label (is_declining) is the only place
April data is used, and it's used correctly: as the outcome to predict,
not as an input. [Add one line here if any correlation above looked
suspiciously high — near 1.0 — as that could indicate an accidental
near-duplicate of the label.]

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
## Claim Rewrite — Safe Language

**Before (overclaiming):** "My model predicts which pages will decline."

**After (safe):** "Within this sample and time window, my model's ranked
output is observed to concentrate more true decliners in its top 50 than
a naive baseline, under a client-grouped validation split. This is a
directional, decision-support signal — not a guarantee about any
individual page, and not evidence of a causal relationship between the
features and decline."

**Before:** "Low CTR causes pages to decline."

**After:** "Low CTR is associated with the declining-page group in this
sample; this is an observed pattern, not a demonstrated causal mechanism."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.